# Notebook 01: Data Exploration & Preprocessing (Stage 1 & Stage 2)

This notebook demonstrates loading social media post data, schema verification, and sequential preprocessing stages:
- **Stage 1**: Schema Ingestion, Field Mapping & Pydantic Validation (`DataCollector`)
- **Stage 2.1**: Deduplication (`remove_duplicates`)
- **Stage 2.2**: Spam & Bot Filtering (`filter_spam_and_bots`)
- **Stage 2.3**: Text Cleaning (HTML, whitespace, URLs)
- **Stage 2.4**: Text Normalization (Unicode NFKC, lowercase, repeated characters)
- **Stage 2.5**: Preserve Semantic Information (Hashtags, Emojis, Negation context)
- **Stage 2.6**: End-to-End `PreprocessingPipeline` Execution & Output Verification

### 1. Imports & Configuration Setup

In [1]:
import sys
import os

# Ensure project root is the current working directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if '.' not in sys.path:
    sys.path.insert(0, '.')

# Ensure UTF-8 output encoding on Windows consoles
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

import pandas as pd
import numpy as np
from loguru import logger

# Stage 01 imports
from src.stage_01_data_collection import (
    DataCollector,
    RawPostSchema,
    SocialDataBatch
)
from src.stage_01_data_collection.collector import standardize_columns

# Stage 02 imports
from src.stage_02_preprocessing import (
    PreprocessingPipeline,
    remove_duplicates,
    filter_spam_and_bots,
    clean_text,
    normalize_text,
    preserve_semantic_information
)

# File I/O utilities
from src.utils.file_io import load_yaml, load_dataframe, save_dataframe

config = load_yaml('configs/config.yaml')
print(f'Working directory: {os.getcwd()}')
print('All modules and configurations successfully imported.')


All modules and configurations successfully imported.


### 2. Inspect Raw Social Media Dataset
Check available raw datasets in `data/01_raw/`.

In [2]:
raw_csv_path = 'data/01_raw/social_listening_30days.csv'
df_raw = load_dataframe(raw_csv_path)
print(f'Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
df_raw.head(3)


Raw dataset shape: 1513 rows, 10 columns


,post_id,text,timestamp,likes,shares,comments,hashtags,followers,platform,topic_category
0,rd_t3_1wthj2j,How to get the Windows 11 2026 Update,2026-09-29 18:10:14,79,10,17,['technology'],1307,Reddit,technology_windows_hardware
1,rd_t3_1ws5o4y,Windows XP IA64 Bit Edition Version 2003 runni...,2026-09-28 05:08:01,19,11,21,['technology'],1070,Reddit,technology_windows_hardware
2,rd_t3_1wryf8u,list of all keyboard shortcuts in Windows. For...,2026-09-27 23:04:15,24,16,27,['technology'],1432,Reddit,technology_windows_hardware


### 3. Stage 1: Data Collection, Ingestion & Pydantic Validation
Standardize schema column aliases and validate via `RawPostSchema`.

In [3]:
collector = DataCollector(output_path='data/01_raw/raw_social_posts.parquet')
df_validated = collector.collect_from_file(raw_csv_path)
print(f'Successfully validated & ingested {len(df_validated)} posts.')
df_validated.info()


2026-09-30 21:02:58.181 | INFO     | src.stage_01_data_collection.collector:collect_from_file:103 - Collecting data from input file: ../data/01_raw/social_listening_30days.csv
2026-09-30 21:02:58.256 | INFO     | src.stage_01_data_collection.collector:validate_and_ingest:97 - Successfully validated 1513 posts. Dropped 0 malformed posts.
2026-09-30 21:02:58.348 | INFO     | src.utils.file_io:save_dataframe:51 - Saved 1513 rows to: ../data/01_raw/raw_social_posts.parquet


Successfully validated & ingested 1513 posts.
<class 'pandas.DataFrame'>
RangeIndex: 1513 entries, 0 to 1512
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   post_id         1513 non-null   str   
 1   text            1513 non-null   str   
 2   timestamp       1513 non-null   str   
 3   likes           1513 non-null   int64 
 4   shares          1513 non-null   int64 
 5   comments        1513 non-null   int64 
 6   hashtags        1513 non-null   object
 7   followers       1513 non-null   int64 
 8   platform        1513 non-null   str   
 9   topic_category  1513 non-null   str   
dtypes: int64(4), object(1), str(5)
memory usage: 512.9+ KB


### 4. Stage 2: Step-by-Step Preprocessing Exploration
#### 4.1 Remove Duplicates

In [4]:
df_dedup = remove_duplicates(df_validated, subset=['text'])
print(f'Before: {len(df_validated)} | After deduplication: {len(df_dedup)}')
df_dedup.head(2)

2026-09-30 21:03:09.972 | INFO     | src.stage_02_preprocessing.deduplicator:remove_duplicates:16 - [Remove Duplicate] Removed 0 duplicate posts. Remaining: 1513 posts.


Before: 1513 | After deduplication: 1513


,post_id,text,timestamp,likes,shares,comments,hashtags,followers,platform,topic_category
0,rd_t3_1wthj2j,How to get the Windows 11 2026 Update,2026-09-29 18:10:14,79,10,17,[technology],1307,Reddit,technology_windows_hardware
1,rd_t3_1ws5o4y,Windows XP IA64 Bit Edition Version 2003 runni...,2026-09-28 05:08:01,19,11,21,[technology],1070,Reddit,technology_windows_hardware


#### 4.2 Spam & Bot Filtering
Filters based on follower thresholds, excessive hashtags, and URL density.

In [5]:
df_filtered = filter_spam_and_bots(df_dedup, min_followers=5, max_hashtags=15, max_urls=4)
print(f'Remaining authentic posts: {len(df_filtered)}')
df_filtered.head(2)

2026-09-30 21:03:17.548 | INFO     | src.stage_02_preprocessing.spam_filter:filter_spam_and_bots:55 - [Remove Spam Bot] Filtered out 0 spam/bot posts. Remaining: 1513 posts.


Remaining authentic posts: 1513


,post_id,text,timestamp,likes,shares,comments,hashtags,followers,platform,topic_category
0,rd_t3_1wthj2j,How to get the Windows 11 2026 Update,2026-09-29 18:10:14,79,10,17,[technology],1307,Reddit,technology_windows_hardware
1,rd_t3_1ws5o4y,Windows XP IA64 Bit Edition Version 2003 runni...,2026-09-28 05:08:01,19,11,21,[technology],1070,Reddit,technology_windows_hardware


#### 4.3 Text Cleaning, Normalization & Semantic Preservation Demonstration

In [6]:
sample_texts = [
    'Check this out!!! <a href="http://example.com">LINK</a> #AI #Tech https://bit.ly/3xyz 🔥🔥',
    'I CANNOT believe this new model... sooo goooood!! 🚀',
    'Standard text normalization test with mixed Việt ngữ & Unicode café/café.'
]

for raw_text in sample_texts:
    cleaned = clean_text(raw_text, remove_url=True)
    normalized = normalize_text(cleaned, form='NFKC', lowercase=True, max_repeats=2)
    preserved = preserve_semantic_information(normalized, demojize=False)
    
    print(f'RAW:        {raw_text}')
    print(f'PROCESSED:  {preserved}')
    print('-' * 70)

RAW:        Check this out!!! <a href="http://example.com">LINK</a> #AI #Tech https://bit.ly/3xyz 🔥🔥
PROCESSED:  check this out!! link  #ai   #tech  🔥🔥
----------------------------------------------------------------------
RAW:        I CANNOT believe this new model... sooo goooood!! 🚀
PROCESSED:  i cannot believe this new model.. soo good!! 🚀
----------------------------------------------------------------------
RAW:        Standard text normalization test with mixed Việt ngữ & Unicode café/café.
PROCESSED:  standard text normalization test with mixed việt ngữ & unicode café/café.
----------------------------------------------------------------------


### 5. Execute Full Preprocessing Pipeline & Save Outputs
Runs the complete Stage 2 pipeline on the validated dataset and saves to `data/02_preprocessed/preprocessed_posts.parquet`.

In [7]:
pipeline = PreprocessingPipeline(config.get('stage_02_preprocessing', {}))
pipeline.output_path = 'data/02_preprocessed/preprocessed_posts.parquet'
df_preprocessed = pipeline.run(df_validated)

print(f'Preprocessing successfully completed! Final dataset shape: {df_preprocessed.shape}')
df_preprocessed[['post_id', 'text', 'cleaned_text', 'likes', 'shares', 'comments']].head(5)


2026-09-30 21:03:33.923 | INFO     | src.stage_02_preprocessing.pipeline:run:37 - Starting data preprocessing on 1513 records...
2026-09-30 21:03:33.926 | INFO     | src.stage_02_preprocessing.deduplicator:remove_duplicates:16 - [Remove Duplicate] Removed 0 duplicate posts. Remaining: 1513 posts.
2026-09-30 21:03:33.931 | INFO     | src.stage_02_preprocessing.spam_filter:filter_spam_and_bots:55 - [Remove Spam Bot] Filtered out 0 spam/bot posts. Remaining: 1513 posts.
2026-09-30 21:03:33.932 | INFO     | src.stage_02_preprocessing.pipeline:run:46 - Executing text cleaning, normalization, and semantic preservation...
2026-09-30 21:03:34.062 | INFO     | src.utils.file_io:save_dataframe:51 - Saved 1513 rows to: ../data/02_preprocessed/preprocessed_posts.parquet
2026-09-30 21:03:34.063 | INFO     | src.stage_02_preprocessing.pipeline:run:53 - Preprocessing completed. Saved clean dataset (1513 rows) to: ../data/02_preprocessed/preprocessed_posts.parquet


Preprocessing successfully completed! Final dataset shape: (1513, 11)


,post_id,text,cleaned_text,likes,shares,comments
0,rd_t3_1wthj2j,How to get the Windows 11 2026 Update,how to get the windows 11 2026 update,79,10,17
1,rd_t3_1ws5o4y,Windows XP IA64 Bit Edition Version 2003 runni...,windows xp ia64 bit edition version 2003 runni...,19,11,21
2,rd_t3_1wryf8u,list of all keyboard shortcuts in Windows. For...,list of all keyboard shortcuts in windows. for...,24,16,27
3,rd_t3_1wruyyr,Do people still listen to windows remixes?. Do...,do people still listen to windows remixes?. do...,82,2,25
4,rd_t3_1wru3fa,2022 image AI tries recreating windows logos,2022 image ai tries recreating windows logos,17,17,24
